# LHS Beam Design: 12 + Optional Extra Samples

Latin hypercube sample over:
- **Web thickness (b):** 1 mm to 8 mm
- **Web height (H):** 12 mm to 23 mm

Generates 12 beams for an initial design. You can then request additional samples that are LHS-spaced and maximally separated from the initial 12.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats.qmc import LatinHypercube

# Design space (mm)
B_WEB = (1.0, 8.0)   # web thickness
H_WEB = (12.0, 23.0) # web height

SEED = 42  # change for a different LHS draw
N_INITIAL = 12

sampler = LatinHypercube(d=2, seed=SEED)
u = sampler.random(n=N_INITIAL)

b_mm = B_WEB[0] + u[:, 0] * (B_WEB[1] - B_WEB[0])
H_mm = H_WEB[0] + u[:, 1] * (H_WEB[1] - H_WEB[0])

df_initial = pd.DataFrame({
    'b_web_mm': np.round(b_mm, 3),
    'H_web_mm': np.round(H_mm, 3),
})
df_initial.index = np.arange(1, len(df_initial) + 1)
df_initial.index.name = 'beam'

print('Initial 12 LHS beams')
print(df_initial)

# Store in normalized [0,1] for later "away from" logic
X_initial = u.copy()

## Additional samples (LHS-spaced away from the 12)

Set `N_EXTRA` to how many more beams you want. The script generates a large LHS pool and selects the `N_EXTRA` points that **maximize the minimum distance** to the existing 12 (and to each other), so new points stay well spread and away from the initial design.

In [ ]:
N_EXTRA = 5  # set to 0 to skip; e.g. 5 or 10 for more runs
POOL_SIZE = 500  # LHS candidates to choose from (increase if N_EXTRA is large)

if N_EXTRA <= 0:
    print('N_EXTRA = 0: no additional samples.')
    df_extra = None
else:
    # New LHS pool (use different seed so we don't repeat initial 12)
    pool_sampler = LatinHypercube(d=2, seed=SEED + 1000)
    u_pool = pool_sampler.random(n=POOL_SIZE)

    # Minimize negative of "minimum distance to existing + to each other"
    def min_dist_to_set(u_new, u_existing):
        d = np.linalg.norm(u_new - u_existing, axis=1)
        return np.min(d)

    chosen = []
    u_used = X_initial.copy()

    for _ in range(N_EXTRA):
        best_idx = None
        best_min_d = -1.0
        for i in range(len(u_pool)):
            u_cand = u_pool[i : i + 1]
            d_to_existing = min_dist_to_set(u_cand, u_used)
            if d_to_existing > best_min_d:
                best_min_d = d_to_existing
                best_idx = i
        if best_idx is None:
            break
        chosen.append(u_pool[best_idx])
        u_used = np.vstack([u_used, u_pool[best_idx : best_idx + 1]])
        # Remove chosen so we don't pick again
        u_pool = np.delete(u_pool, best_idx, axis=0)

    u_extra = np.array(chosen)
    b_extra = B_WEB[0] + u_extra[:, 0] * (B_WEB[1] - B_WEB[0])
    H_extra = H_WEB[0] + u_extra[:, 1] * (H_WEB[1] - H_WEB[0])

    df_extra = pd.DataFrame({
        'b_web_mm': np.round(b_extra, 3),
        'H_web_mm': np.round(H_extra, 3),
    })
    df_extra.index = np.arange(N_INITIAL + 1, N_INITIAL + len(df_extra) + 1)
    df_extra.index.name = 'beam'

    print(f'Additional {N_EXTRA} beams (maximin from LHS pool)')
    print(df_extra)

In [ ]:
# Optional: plot and export
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.scatter(df_initial['b_web_mm'], df_initial['H_web_mm'], c='C0', s=80, label='Initial 12', zorder=5)
if df_extra is not None and len(df_extra) > 0:
    ax.scatter(df_extra['b_web_mm'], df_extra['H_web_mm'], c='C1', s=80, marker='s', label=f'Extra {len(df_extra)}', zorder=5)
ax.set_xlabel('Web thickness b (mm)')
ax.set_ylabel('Web height H (mm)')
ax.set_title('LHS beam design (normalized space)')
ax.legend()
ax.set_xlim(B_WEB[0] - 0.2, B_WEB[1] + 0.2)
ax.set_ylim(H_WEB[0] - 0.5, H_WEB[1] + 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Export CSVs
df_initial.to_csv('LHS_beams_initial_12.csv')
print('Saved: LHS_beams_initial_12.csv')
if df_extra is not None and len(df_extra) > 0:
    df_extra.to_csv('LHS_beams_extra.csv')
    print('Saved: LHS_beams_extra.csv')
    df_all = pd.concat([df_initial, df_extra])
    df_all.to_csv('LHS_beams_all.csv')
    print('Saved: LHS_beams_all.csv')